# Fisher-KPP PINN Training Notebook

## 1. Import Libraries and Configure Plotting

In [56]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import time
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt

## 2. Define Device Selection Utility

In [57]:
def get_preferred_device():
    """Use CUDA when available, otherwise fall back to CPU."""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


device = get_preferred_device()
print(f"Using device: {device}")

Using device: cuda


## 3. Define the Exact Fisher-KPP Solution

In [58]:
def exact_solution(x, t, D=0.01, R=1.0):
    """Analytical traveling-wave solution used as the target reference."""
    sqrt_term = np.sqrt(R / (2.0 * D))
    wave_speed = np.sqrt(2.0 * D * R)
    return 1.0 / (1.0 + np.exp(sqrt_term * (x - wave_speed * t)))

## 4. Build the PINN Model Class

Create the `PINN_FisherKPP` class with the 7-hidden-layer Tanh architecture, Xavier initialization, adaptive loss weights, training history containers, and the training/checkpoint utilities needed by the workflow.

In [59]:
class PINN_FisherKPP(nn.Module):
    """7 hidden layers x 50 neurons with Tanh activations for Fisher-KPP."""

    def __init__(self, layers=(2, 50, 50, 50, 50, 50, 50, 50, 1),
                 D=0.01, R=1.0, adaptive_weights=True):
        super().__init__()
        self.D = D
        self.R = R
        self.adaptive_weights = adaptive_weights
        self.lambda_max = 10_000.0
        self.lambda_ic = 1.0
        self.lambda_bc = 1.0
        self.lambda_res = 1.0

        modules = []
        for i in range(len(layers) - 1):
            linear = nn.Linear(layers[i], layers[i + 1])
            nn.init.xavier_normal_(linear.weight)
            nn.init.zeros_(linear.bias)
            modules.append(linear)
            if i < len(layers) - 2:
                modules.append(nn.Tanh())
        self.net = nn.Sequential(*modules)

        self.loss_history = []
        self.loss_ic_history = []
        self.loss_bc_history = []
        self.loss_res_history = []
        self.l2_error_history = []
        self.iteration_history = []
        self._global_iter = 0

        n = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print("=" * 65)
        print(f"  PINN  7x50 | Tanh | Xavier | {n:,} parameters")
        print(f"  D={D}  R={R}  lambda_max={self.lambda_max}")
        print("=" * 65)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))

    def pde_residual(self, x_r, t_r):
        u = self.forward(x_r, t_r)
        u_t = torch.autograd.grad(u, t_r, torch.ones_like(u), create_graph=True)[0]
        u_x = torch.autograd.grad(u, x_r, torch.ones_like(u), create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x_r, torch.ones_like(u_x), create_graph=True)[0]
        return u_t - self.D * u_xx - self.R * u * (1.0 - u)

    def compute_loss(self, x_ic, t_ic, u_ic,
                     x_bc, t_bc, u_bc,
                     x_r, t_r):
        l_ic = torch.mean((self.forward(x_ic, t_ic) - u_ic) ** 2)
        l_bc = torch.mean((self.forward(x_bc, t_bc) - u_bc) ** 2)
        l_res = torch.mean(self.pde_residual(x_r, t_r) ** 2)
        loss = self.lambda_ic * l_ic + self.lambda_bc * l_bc + self.lambda_res * l_res
        return loss, l_ic, l_bc, l_res

    def _update_adaptive_weights(self, loss_ic, loss_bc, loss_res):
        eps = 1e-10
        l_res = loss_res.item()
        self.lambda_ic = min(l_res / (loss_ic.item() + eps), self.lambda_max)
        self.lambda_bc = min(l_res / (loss_bc.item() + eps), self.lambda_max)

    @torch.no_grad()
    def compute_l2_error(self, x_test, t_test, u_exact_np):
        pred = self.forward(x_test, t_test).detach().cpu().numpy()
        return np.linalg.norm(pred - u_exact_np) / np.linalg.norm(u_exact_np)

    def train_adam(self,
                   x_ic, t_ic, u_ic,
                   x_bc, t_bc, u_bc,
                   x_r_raw, t_r_raw,
                   iterations=10_000,
                   learning_rate=1e-3,
                   decay_rate=0.99,
                   sched_step_every=100,
                   print_every=1000,
                   phase_name="Adam Training",
                   optimizer=None,
                   lr_schedule="exponential",
                   warmup_iters=0):
        print(f"\n{'='*65}")
        print(f"  {phase_name}")
        optimizer_state = "fresh" if optimizer is None else "restored"
        print(f"  optimizer : Adam [{optimizer_state}]  lr={learning_rate}")
        print(f"  schedule  : {lr_schedule}")
        if lr_schedule == "exponential":
            print(f"             decay={decay_rate} every {sched_step_every} iters")
        elif lr_schedule == "cosine":
            print(f"             cosine annealing over {iterations} iters")
        elif lr_schedule == "linear":
            print(f"             linear decay over {iterations} iters")
        elif lr_schedule == "exponential_delayed":
            print(f"             {warmup_iters} iter warmup, then decay={decay_rate}")
        print(f"  iterations: {iterations}")
        print(f"{'='*65}")

        x_np = np.linspace(0, 1, 201).reshape(-1, 1).astype(np.float32)
        t_np = np.ones_like(x_np)
        u_ex = exact_solution(x_np, t_np, self.D, self.R)
        model_device = next(self.parameters()).device
        x_test = torch.from_numpy(x_np).to(model_device)
        t_test = torch.from_numpy(t_np).to(model_device)

        x_r = x_r_raw.clone().detach().requires_grad_(True)
        t_r = t_r_raw.clone().detach().requires_grad_(True)

        if optimizer is None:
            optimizer = optim.Adam(self.parameters(), lr=learning_rate)
        else:
            for param_group in optimizer.param_groups:
                param_group['lr'] = learning_rate

        if lr_schedule == "exponential":
            scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=decay_rate)
            use_scheduler = True
        elif lr_schedule == "cosine":
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=iterations, eta_min=1e-7)
            use_scheduler = True
        elif lr_schedule == "linear":
            scheduler = optim.lr_scheduler.LinearLR(
                optimizer, start_factor=1.0, total_iters=iterations, last_epoch=-1)
            use_scheduler = True
        elif lr_schedule == "exponential_delayed":
            scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=decay_rate)
            use_scheduler = True
            warmup_iters_actual = warmup_iters
        else:
            use_scheduler = False
            print(f"  WARNING: Unknown schedule '{lr_schedule}', using no schedule")

        print(f"\n  {'Iter':>7}  {'Loss':>11}  {'L_IC':>11}  {'L_BC':>11}  {'L_Res':>11}  {'L2_Err':>11}  {'lam_IC':>9}  {'lam_BC':>9}  {'LR':>9}  {'Time':>7}")
        print("  " + "-" * 105)

        t0 = time.time()
        l2 = self.compute_l2_error(x_test, t_test, u_ex)

        for it in range(iterations):
            optimizer.zero_grad()
            loss, l_ic, l_bc, l_res = self.compute_loss(
                x_ic, t_ic, u_ic, x_bc, t_bc, u_bc, x_r, t_r)
            loss.backward()
            optimizer.step()

            if use_scheduler:
                if lr_schedule == "exponential":
                    if (it + 1) % sched_step_every == 0:
                        scheduler.step()
                elif lr_schedule == "exponential_delayed":
                    if it >= warmup_iters_actual and (it + 1) % sched_step_every == 0:
                        scheduler.step()
                else:
                    scheduler.step()

            if self.adaptive_weights and (it + 1) % 100 == 0:
                self._update_adaptive_weights(l_ic, l_bc, l_res)

            if it % print_every == 0 or it == iterations - 1:
                l2 = self.compute_l2_error(x_test, t_test, u_ex)
                cur_lr = optimizer.param_groups[0]['lr']
                elapsed = time.time() - t0

                self.loss_history.append(loss.item())
                self.loss_ic_history.append(l_ic.item())
                self.loss_bc_history.append(l_bc.item())
                self.loss_res_history.append(l_res.item())
                self.l2_error_history.append(l2)
                self.iteration_history.append(self._global_iter + it)

                print(f"  {it:7d}  {loss.item():11.3e}  {l_ic.item():11.3e}  {l_bc.item():11.3e}  {l_res.item():11.3e}  {l2:11.3e}  {self.lambda_ic:9.1f}  {self.lambda_bc:9.1f}  {cur_lr:9.2e}  {elapsed:6.1f}s")

        self._global_iter += iterations
        print(f"\n  ✓ Adam done — final L2 = {l2:.6e}")
        return l2, optimizer

    def train_lbfgs(self,
                    x_ic, t_ic, u_ic,
                    x_bc, t_bc, u_bc,
                    x_r_raw, t_r_raw,
                    outer_steps=500,
                    max_iter=20,
                    history_size=50,
                    tolerance_grad=1e-7,
                    tolerance_change=1e-9,
                    print_every=50,
                    phase_name="L-BFGS Fine-Tuning",
                    optimizer=None):
        print(f"\n{'='*65}")
        print(f"  {phase_name}")
        optimizer_state = "fresh" if optimizer is None else "restored"
        print(f"  optimizer    : L-BFGS [{optimizer_state}]  lr=1.0 (strong Wolfe line search)")
        print(f"  outer_steps  : {outer_steps}")
        print(f"  max_iter     : {max_iter}  (internal evals per outer step)")
        print(f"  history_size : {history_size}")
        print(f"  Starting from the current model weights")
        print(f"  (L-BFGS state can be restored from checkpoint if available)")
        print(f"{'='*65}")

        x_np = np.linspace(0, 1, 201).reshape(-1, 1).astype(np.float32)
        t_np = np.ones_like(x_np)
        u_ex = exact_solution(x_np, t_np, self.D, self.R)
        model_device = next(self.parameters()).device
        x_test = torch.from_numpy(x_np).to(model_device)
        t_test = torch.from_numpy(t_np).to(model_device)

        x_r = x_r_raw.clone().detach().requires_grad_(True)
        t_r = t_r_raw.clone().detach().requires_grad_(True)

        if optimizer is None:
            optimizer = optim.LBFGS(
                self.parameters(),
                lr=1.0,
                max_iter=max_iter,
                history_size=history_size,
                tolerance_grad=tolerance_grad,
                tolerance_change=tolerance_change,
                line_search_fn='strong_wolfe')

        loss_cache = [None, None, None, None]

        def closure():
            optimizer.zero_grad()
            loss, l_ic, l_bc, l_res = self.compute_loss(
                x_ic, t_ic, u_ic, x_bc, t_bc, u_bc, x_r, t_r)
            loss.backward()
            loss_cache[0] = loss
            loss_cache[1] = l_ic
            loss_cache[2] = l_bc
            loss_cache[3] = l_res
            return loss

        print(f"\n  {'Step':>6}  {'Loss':>11}  {'L_IC':>11}  {'L_BC':>11}  {'L_Res':>11}  {'L2_Err':>11}  {'lam_IC':>9}  {'lam_BC':>9}  {'Time':>7}")
        print("  " + "-" * 97)

        t0 = time.time()
        l2 = self.compute_l2_error(x_test, t_test, u_ex)

        for step in range(outer_steps):
            optimizer.step(closure)

            if self.adaptive_weights and (step + 1) % 20 == 0:
                if loss_cache[1] is not None:
                    self._update_adaptive_weights(
                        loss_cache[1], loss_cache[2], loss_cache[3])

            if step % print_every == 0 or step == outer_steps - 1:
                l2 = self.compute_l2_error(x_test, t_test, u_ex)
                elapsed = time.time() - t0
                loss = loss_cache[0]
                l_ic = loss_cache[1]
                l_bc = loss_cache[2]
                l_res = loss_cache[3]

                self.loss_history.append(loss.item())
                self.loss_ic_history.append(l_ic.item())
                self.loss_bc_history.append(l_bc.item())
                self.loss_res_history.append(l_res.item())
                self.l2_error_history.append(l2)
                self.iteration_history.append(self._global_iter + step * max_iter)

                print(f"  {step:6d}  {loss.item():11.3e}  {l_ic.item():11.3e}  {l_bc.item():11.3e}  {l_res.item():11.3e}  {l2:11.3e}  {self.lambda_ic:9.1f}  {self.lambda_bc:9.1f}  {elapsed:6.1f}s")

        self._global_iter += outer_steps * max_iter
        print(f"\n  ✓ L-BFGS done — final L2 = {l2:.6e}")
        return l2, optimizer

    def save_checkpoint(self, path, optimizer=None):
        ck = {
            'state_dict': self.state_dict(),
            'lambda_ic': self.lambda_ic,
            'lambda_bc': self.lambda_bc,
            'lambda_res': self.lambda_res,
            'loss_history': self.loss_history,
            'loss_ic_history': self.loss_ic_history,
            'loss_bc_history': self.loss_bc_history,
            'loss_res_history': self.loss_res_history,
            'l2_error_history': self.l2_error_history,
            'iteration_history': self.iteration_history,
            'global_iter': self._global_iter,
        }
        if optimizer is not None:
            ck['optimizer_state'] = optimizer.state_dict()
        torch.save(ck, path)
        print(f"  Checkpoint saved -> {path}")
        if optimizer is not None:
            print("    (includes optimizer state for resumable training)")

    def load_checkpoint(self, path, optimizer=None):
        ck = torch.load(path, map_location=next(self.parameters()).device)
        self.load_state_dict(ck['state_dict'])
        self.lambda_ic = ck['lambda_ic']
        self.lambda_bc = ck['lambda_bc']
        self.lambda_res = ck['lambda_res']
        self.loss_history = ck['loss_history']
        self.loss_ic_history = ck['loss_ic_history']
        self.loss_bc_history = ck['loss_bc_history']
        self.loss_res_history = ck['loss_res_history']
        self.l2_error_history = ck['l2_error_history']
        self.iteration_history = ck['iteration_history']
        self._global_iter = ck.get('global_iter', 0)
        print(f"  Checkpoint loaded <- {path}")
        if optimizer is not None and 'optimizer_state' in ck:
            optimizer.load_state_dict(ck['optimizer_state'])
            print("    (optimizer state restored)")
        elif optimizer is not None and 'optimizer_state' not in ck:
            print("    (no optimizer state in checkpoint)")

    def plot_history(self, save_path='training_history_lbfgs.png'):
        iters = np.array(self.iteration_history)
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        axes[0, 0].semilogy(iters, self.loss_history, 'b-', lw=1.5)
        axes[0, 0].set(xlabel='Iteration', ylabel='Total Loss', title='Total Loss (log scale)')
        axes[0, 0].grid(alpha=0.3)

        axes[0, 1].semilogy(iters, self.l2_error_history, 'r-', lw=2, label='PINN (Adam+L-BFGS)')
        axes[0, 1].axhline(1.42e-4, color='k', ls=':', lw=1.5, label='FDM  1.42x10^-4')
        axes[0, 1].axhline(5.57e-2, color='g', ls='--', lw=1.5, label='Paper best  5.57%')
        axes[0, 1].axhline(9.794e-2, color='orange', ls='--', lw=1.5, label='Paper retrain  9.79%')
        axes[0, 1].set(xlabel='Iteration', ylabel='Relative L2 Error', title='L2 Error vs. Iteration')
        axes[0, 1].legend(fontsize=8)
        axes[0, 1].grid(alpha=0.3)

        axes[1, 0].semilogy(iters, self.loss_ic_history, label='IC')
        axes[1, 0].semilogy(iters, self.loss_bc_history, label='BC')
        axes[1, 0].semilogy(iters, self.loss_res_history, label='Residual')
        axes[1, 0].set(xlabel='Iteration', ylabel='Loss', title='Loss Components')
        axes[1, 0].legend()
        axes[1, 0].grid(alpha=0.3)

        axes[1, 1].plot(iters, self.l2_error_history, 'b-', lw=1.5)
        if np.any(iters >= 10_000):
            axes[1, 1].axvline(10_000, color='purple', ls=':', lw=1.5, label='Adam -> L-BFGS')
        if np.any(iters >= 20_000):
            axes[1, 1].axvline(20_000, color='orange', ls='--', lw=1, label='L-BFGS Phase 2')
        axes[1, 1].set(xlabel='Iteration', ylabel='L2 Error', title='Training Phases')
        axes[1, 1].legend()
        axes[1, 1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  Plot saved -> {save_path}")

## 12. Load and Prepare Dataset

Load `data_samples.npz`, extract collocation, initial, and boundary points, and compute target values from the exact solution.

In [60]:
def prepare_data(path):
    """Load the dataset and build tensors for PINN training."""
    d = np.load(path)
    print(f"\nDataset loaded from '{path}'")
    print(f"  Collocation points : {d['collocation'].shape[0]:,}  (paper: 10,000)")
    print(f"  Initial cond. pts  : {d['initial'].shape[0]:,}  (paper:  1,000)")
    print(f"  Boundary cond. pts : {d['boundary'].shape[0]:,}  (paper:  2,000 = 1,000 per side)")

    def T(a):
        return torch.tensor(a, dtype=torch.float32)

    x_r = T(d['collocation'][:, 0:1])
    t_r = T(d['collocation'][:, 1:2])
    x_ic = T(d['initial'][:, 0:1])
    t_ic = T(d['initial'][:, 1:2])
    x_bc = T(d['boundary'][:, 0:1])
    t_bc = T(d['boundary'][:, 1:2])

    u_ic = T(exact_solution(x_ic.numpy(), t_ic.numpy()))
    u_bc = T(exact_solution(x_bc.numpy(), t_bc.numpy()))

    return x_ic, t_ic, u_ic, x_bc, t_bc, u_bc, x_r, t_r

## 13. Set Seeds and Move Data to Device

Set deterministic random seeds, load the dataset, and move all tensors to the selected device.

In [61]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

x_ic, t_ic, u_ic, x_bc, t_bc, u_bc, x_r, t_r = prepare_data('data_samples.npz')

x_ic = x_ic.to(device)
t_ic = t_ic.to(device)
u_ic = u_ic.to(device)
x_bc = x_bc.to(device)
t_bc = t_bc.to(device)
u_bc = u_bc.to(device)
x_r = x_r.to(device)
t_r = t_r.to(device)

print(f"\nTensors moved to: {device}")


Dataset loaded from 'data_samples.npz'
  Collocation points : 10,000  (paper: 10,000)
  Initial cond. pts  : 1,000  (paper:  1,000)
  Boundary cond. pts : 2,000  (paper:  2,000 = 1,000 per side)

Tensors moved to: cuda


## 14. Instantiate the PINN Model

Create the `PINN_FisherKPP` instance with the desired architecture, PDE constants, and adaptive weighting enabled.

In [62]:
pinn = PINN_FisherKPP(
    layers=[2, 50, 50, 50, 50, 50, 50, 50, 1],
    D=0.01,
    R=1.0,
    adaptive_weights=True,
).to(device)

# Common data dictionary for all phases
train_data = dict(
    x_ic=x_ic, t_ic=t_ic, u_ic=u_ic,
    x_bc=x_bc, t_bc=t_bc, u_bc=u_bc,
    x_r_raw=x_r, t_r_raw=t_r,
)

  PINN  7x50 | Tanh | Xavier | 15,501 parameters
  D=0.01  R=1.0  lambda_max=10000.0


## 64-Combination Decay Policy Sweep

Run all 64 schedule combinations across three Adam phases (4^3) and compare final L2 error and total training time.

Schedules used:
- `exponential`
- `exponential_delayed`
- `cosine`
- `linear`

In [63]:
# import itertools
# import pandas as pd

# # 64-combination sweep configuration
# policies = ["exponential", "exponential_delayed", "cosine", "linear"]
# all_combinations = list(itertools.product(policies, repeat=3))  # 4^3 = 64

# # Keep these at paper-scale for full benchmark.
# # Warning: this can take hours depending on GPU/CPU.
# SWEEP_ITERS_P1 = 10_000
# SWEEP_ITERS_P2 = 5_000
# SWEEP_ITERS_P3 = 5_000
# SWEEP_WARMUP = 1_000
# SWEEP_DECAY = 0.99
# SWEEP_STEP_EVERY = 100

# print(f"Total combinations: {len(all_combinations)}")
# print("Starting 64-run benchmark sweep...")

# sweep_results = []
# sweep_t0 = time.time()

# for idx, (sched1, sched2, sched3) in enumerate(all_combinations, start=1):
#     combo_t0 = time.time()

#     # Reset seeds per run for fair comparison
#     torch.manual_seed(42)
#     np.random.seed(42)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(42)

#     model = PINN_FisherKPP(
#         layers=[2, 50, 50, 50, 50, 50, 50, 50, 1],
#         D=0.01,
#         R=1.0,
#         adaptive_weights=True,
#     ).to(device)

#     run_data = dict(
#         x_ic=x_ic, t_ic=t_ic, u_ic=u_ic,
#         x_bc=x_bc, t_bc=t_bc, u_bc=u_bc,
#         x_r_raw=x_r, t_r_raw=t_r,
#     )

#     # Phase 1
#     l2_p1, opt = model.train_adam(
#         **run_data,
#         iterations=SWEEP_ITERS_P1,
#         learning_rate=1e-3,
#         decay_rate=SWEEP_DECAY,
#         sched_step_every=SWEEP_STEP_EVERY,
#         print_every=SWEEP_ITERS_P1,
#         phase_name=f"Sweep Run {idx}/64 - Phase 1",
#         lr_schedule=sched1,
#         warmup_iters=SWEEP_WARMUP,
#     )

#     # Phase 2 (preserve Adam state)
#     l2_p2, opt = model.train_adam(
#         **run_data,
#         iterations=SWEEP_ITERS_P2,
#         learning_rate=1e-4,
#         decay_rate=SWEEP_DECAY,
#         sched_step_every=SWEEP_STEP_EVERY,
#         print_every=SWEEP_ITERS_P2,
#         phase_name=f"Sweep Run {idx}/64 - Phase 2",
#         optimizer=opt,
#         lr_schedule=sched2,
#         warmup_iters=SWEEP_WARMUP,
#     )

#     # Phase 3 (preserve Adam state)
#     l2_p3, opt = model.train_adam(
#         **run_data,
#         iterations=SWEEP_ITERS_P3,
#         learning_rate=1e-5,
#         decay_rate=SWEEP_DECAY,
#         sched_step_every=SWEEP_STEP_EVERY,
#         print_every=SWEEP_ITERS_P3,
#         phase_name=f"Sweep Run {idx}/64 - Phase 3",
#         optimizer=opt,
#         lr_schedule=sched3,
#         warmup_iters=SWEEP_WARMUP,
#     )

#     elapsed = time.time() - combo_t0

#     sweep_results.append({
#         "combo_id": idx,
#         "phase1_policy": sched1,
#         "phase2_policy": sched2,
#         "phase3_policy": sched3,
#         "l2_phase1": float(l2_p1),
#         "l2_phase2": float(l2_p2),
#         "l2_phase3": float(l2_p3),
#         "time_sec": elapsed,
#         "time_min": elapsed / 60.0,
#     })

#     print(
#         f"Completed {idx:02d}/64 | ({sched1}, {sched2}, {sched3}) | "
#         f"L2@P3={l2_p3:.6e} | {elapsed/60.0:.2f} min"
#     )

# sweep_df = pd.DataFrame(sweep_results)
# sweep_df = sweep_df.sort_values(by=["l2_phase3", "time_sec"], ascending=[True, True]).reset_index(drop=True)

# total_elapsed = time.time() - sweep_t0
# print("\nSweep complete.")
# print(f"Total elapsed: {total_elapsed/60.0:.2f} min")

# csv_path = "decay_policy_64_results.csv"
# sweep_df.to_csv(csv_path, index=False)
# print(f"Saved results: {csv_path}")

# display(sweep_df.head(20))

# best_row = sweep_df.iloc[0]
# print("\nBest combination by final L2:")
# print(
#     f"  ({best_row['phase1_policy']}, {best_row['phase2_policy']}, {best_row['phase3_policy']}) "
#     f"-> L2={best_row['l2_phase3']:.6e}, time={best_row['time_min']:.2f} min"
# )

In [64]:
# # Visualization: compare all 64 runs by error and time
# fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# # Left: sorted final L2 per combination
# axes[0].plot(np.arange(1, len(sweep_df) + 1), sweep_df["l2_phase3"].values, marker="o", ms=3)
# axes[0].set_title("Final L2 Across 64 Decay-Policy Combinations")
# axes[0].set_xlabel("Rank (best to worst)")
# axes[0].set_ylabel("Final L2 (Phase 3)")
# axes[0].grid(alpha=0.3)

# # Right: runtime vs final L2
# axes[1].scatter(sweep_df["time_min"], sweep_df["l2_phase3"], alpha=0.8)
# axes[1].set_title("Runtime vs Final L2")
# axes[1].set_xlabel("Time (minutes)")
# axes[1].set_ylabel("Final L2 (Phase 3)")
# axes[1].grid(alpha=0.3)

# plt.tight_layout()
# plt.savefig("decay_policy_64_comparison.png", dpi=150, bbox_inches="tight")
# plt.show()

# print("Saved figure: decay_policy_64_comparison.png")
# print("Saved table : decay_policy_64_results.csv")

## 15. Run Training Phases and Save Checkpoints

Execute Phase 1 and Phase 2 with Adam (restoring optimizer state), then run Phase 3 with L-BFGS fine-tuning and save final checkpoints.

In [ ]:
# Phase 1: Adam with cosine annealing for a smoother initial descent
phase1_l2, optimizer1 = pinn.train_adam(
    **train_data,
    iterations=10_000,
    learning_rate=1e-3,
    decay_rate=0.99,
    sched_step_every=100,
    print_every=1000,
    phase_name="Phase 1 - Adam (10k iters, lr=1e-3)",
    lr_schedule="cosine",
)

pinn.save_checkpoint('ckpt_after_adam.pth', optimizer=optimizer1)

# Phase 2: restore Adam optimizer state and continue at a lower learning rate
checkpoint = torch.load('ckpt_after_adam.pth', map_location=device, weights_only=False)
optimizer2 = optim.Adam(pinn.parameters(), lr=1e-4)
if 'optimizer_state' in checkpoint:
    optimizer2.load_state_dict(checkpoint['optimizer_state'])
    print("\n  Optimizer state restored from checkpoint")

phase2_l2, optimizer2 = pinn.train_adam(
    **train_data,
    iterations=5_000,
    learning_rate=1e-4,
    decay_rate=0.99,
    sched_step_every=100,
    print_every=1000,
    phase_name="Phase 2 - Adam Continue (5k more iters, lr=1e-4)",
    optimizer=optimizer2,
    lr_schedule="cosine",
)

pinn.save_checkpoint('ckpt_after_phase2.pth', optimizer=optimizer2)

# Phase 3: switch to L-BFGS for final polishing
phase3_l2, optimizer3 = pinn.train_lbfgs(
    **train_data,
    outer_steps=500,
    max_iter=20,
    history_size=50,
    tolerance_grad=1e-7,
    tolerance_change=1e-9,
    print_every=50,
    phase_name="Phase 3 - L-BFGS Fine-Tuning",
)

# Final checkpoint (model + histories + LBFGS optimizer state)
pinn.save_checkpoint('ckpt_final_lbfgs.pth', optimizer=optimizer3)


  Phase 1 - Adam (10k iters, lr=1e-3)
  optimizer : Adam [fresh]  lr=0.001
  schedule  : cosine
             cosine annealing over 10000 iters
  iterations: 10000

     Iter         Loss         L_IC         L_BC        L_Res       L2_Err     lam_IC     lam_BC         LR     Time
  ---------------------------------------------------------------------------------------------------------
        0    2.174e-01    2.864e-02    1.827e-01    6.031e-03    7.872e-01        1.0        1.0   1.00e-03     0.0s
     1000    9.374e-04    2.675e-07    5.178e-07    3.131e-04    6.709e-02     1167.2      602.4   9.75e-04    13.3s
     2000    1.044e-03    6.275e-07    1.904e-06    3.484e-04    7.521e-02      554.2      182.8   9.04e-04    24.3s
     3000    8.737e-04    4.915e-07    1.384e-06    2.916e-04    8.138e-02      592.1      210.3   7.94e-04    32.3s
     4000    9.142e-04    7.481e-07    8.093e-07    3.051e-04    8.510e-02      407.2      376.3   6.54e-04    39.8s
     5000    1.366e-03   

## 16. Print Final Metrics and Save Results

Print the final L2 error summary, generate the training plot, and save the collected results to `pinn_results_lbfgs.npz`.

In [ ]:
print("\n" + "=" * 65)
print("  FINAL RESULTS SUMMARY")
print("=" * 65)
print(f"\n  {'Method':<40}  {'L2 Error':>12}  {'vs Paper':>10}")
print("  " + "-" * 65)

rows = [
    ("Phase 1 - Adam (10k iters)", phase1_l2, 5.57e-2, "Paper Phase 1"),
    ("Phase 2 - Adam Continue (5k iters)", phase2_l2, 9.796e-2, "Paper Retrain 1"),
    ("Phase 3 - L-BFGS Fine-Tuning", phase3_l2, 9.794e-2, "Paper Retrain 2"),
]
for label, err, ref, ref_label in rows:
    direction = "BETTER" if err < ref else "worse"
    print(f"  {label:<40}  {err:12.4e}  {direction}")
    print(f"  {'':40}  paper ({ref_label}): {ref:.4e}")
    print()

print("  FDM reference          : 1.42e-4")
print("\n  Key result: model weights carry from phases 1-2, then")
print("  Phase 3 saves and restores L-BFGS optimizer state.")

pinn.plot_history(save_path='training_history_lbfgs.png')

np.savez(
    'pinn_results_lbfgs.npz',
    l2_phase1=phase1_l2,
    l2_phase2=phase2_l2,
    l2_phase3=phase3_l2,
    loss_history=pinn.loss_history,
    l2_error_history=pinn.l2_error_history,
    iteration_history=pinn.iteration_history,
)
print("\nResults saved to pinn_results_lbfgs.npz")
print("\nDone - Adam (phases 1-2) + L-BFGS (phase 3).")


  FINAL RESULTS SUMMARY

  Method                                        L2 Error    vs Paper
  -----------------------------------------------------------------
  Phase 1 - Adam (10k iters)                  5.8205e-02  worse
                                            paper (Paper Phase 1): 5.5700e-02

  Phase 2 - Adam Continue (5k iters)          4.5549e-02  BETTER
                                            paper (Paper Retrain 1): 9.7960e-02

  Phase 3 - L-BFGS Fine-Tuning                4.5550e-02  BETTER
                                            paper (Paper Retrain 2): 9.7940e-02

  FDM reference          : 1.42e-4

  Key result: model weights carry from phases 1-2, then
  Phase 3 saves and restores L-BFGS optimizer state.
  Plot saved -> training_history_lbfgs.png

Results saved to pinn_results_lbfgs.npz

Done - Adam (phases 1-2) + L-BFGS (phase 3).
